In [ ]:
# Cell 1: Import libraries and configure dependencies
# ==============================================================
# LSTM-RNN From Scratch for RML2018.01A Dataset (24 Classes)
# ==============================================================

import h5py, ast, numpy as np, tensorflow as tf, matplotlib.pyplot as plt, seaborn as sns
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path
import os

# --------------------------------------------------------------
# Resolve paths relative to project root
# --------------------------------------------------------------
notebook_dir = Path().resolve()
project_root = notebook_dir.parent  # notebooks/ → repo root

data_root = project_root / "data" / "RML2018" 
model_out = project_root / "models" / "rml2018" / "rml2018_lstm_rnn.keras"

print("Notebook:", notebook_dir)
print("Data directory:", data_root)
print("Model will be saved to:", model_out)

# -------------------- Parameters --------------------
SNR_RANGE = (12, 20)
MAX_SAMPLES_PER_CLASS = 3000
EPOCHS = 250
BATCH_SIZE = 64
LR = 1e-4

# -------------------- Load Data ---------------------
def load_rml2018(h5_path, classes_txt, snr_range=(12,20)):
    with open(classes_txt, "r") as f:
        class_list = ast.literal_eval(f.read().split("=")[-1].strip())
        class_list = [c.strip(" '") for c in class_list]

    with h5py.File(h5_path, "r") as f:
        X, Y, Z = f["X"][:], f["Y"][:], f["Z"][:]

    per_class = {cls: [] for cls in class_list}
    for i in range(len(X)):
        snr = int(Z[i][0])
        if snr_range[0] <= snr <= snr_range[1]:
            label_idx = int(Y[i].argmax())
            cls = class_list[label_idx]
            sig = np.hstack([X[i], np.full((1024,1), snr, dtype=np.float32)])
            per_class[cls].append(sig)

    # Trim to max samples
    for k in per_class:
        per_class[k] = per_class[k][:MAX_SAMPLES_PER_CLASS]

    return per_class

# Paths updated here:
h5_file = data_root / "GOLD_XYZ_OSC.0001_1024.hdf5"
classes_file = data_root / "classes.txt"

data = load_rml2018(h5_file, classes_file)

# -------------------- Prepare Dataset ---------------------
X, y = [], []
for cls, samples in data.items():
    X.extend(samples)
    y.extend([cls] * len(samples))

X = np.array(X, dtype=np.float32)
y = np.array(y)

le = LabelEncoder()
y_enc = le.fit_transform(y)
n_classes = len(le.classes_)
print("Classes:", le.classes_)

# -------------------- Split Dataset ---------------------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)
print("Train/Test split:", X_tr.shape, X_te.shape)

# -------------------- Build Model ---------------------
def build_model(input_shape, n_classes, lr=1e-4):
    model = Sequential([
        LSTM(128, input_shape=input_shape, return_sequences=True),
        Dropout(0.5),
        LSTM(128, return_sequences=False),
        Dropout(0.2),
        Dense(128, activation="relu"),
        Dropout(0.1),
        Dense(n_classes, activation="softmax")
    ])
    model.compile(
        optimizer=Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_model((1024, 3), n_classes, LR)
model.summary()

# -------------------- Train Model ---------------------
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_te, y_te),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2
)

# -------------------- Save Model ---------------------
model.save(model_out)
print("Model saved to:", model_out)

# -------------------- Evaluate ---------------------
y_prob = model.predict(X_te, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("\nClassification Report:")
print(classification_report(y_te, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_te, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=False, cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix – RML2018 (24 Classes)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# ------------------------------
# SAVE TRAINED MODEL ONLY
# ------------------------------

project_root = notebook_dir.parent
model_out = project_root / "models" / "rml2018" / "rml2018_lstm_balanced.keras"

print("Saving model to:", model_out)
model.save(model_out)
print("Done.")


Notebook: /home/rameyjm7/workspace/ML-wireless-signal-classification/notebooks
Data root: /home/rameyjm7/workspace/ML-wireless-signal-classification/data/RML2018
Default model: /home/rameyjm7/workspace/ML-wireless-signal-classification/models/rml2018_lstm_rnn.keras
Checkpoint directory: /home/rameyjm7/workspace/ML-wireless-signal-classification/models/rml2018_checkpoints


In [12]:
# Cell 2: Import libraries and configure dependencies
# =====================================================================
# ONE CELL: FINE-TUNE DEFAULT MODEL + CHECKPOINT + METADATA + BEST PICK
# =====================================================================
import ast, h5py, numpy as np, random, json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
notebook_dir = Path().resolve()
project_root = notebook_dir.parent

data_dir = project_root / "data" / "RML2018"
H5_PATH = data_dir / "GOLD_XYZ_OSC.0001_1024.hdf5"
CLASSES_PATH = data_dir / "classes.txt"

DEFAULT_MODEL = project_root / "models" / "rml2018" / "rml2018_lstm_rnn.keras"

ckpt_dir = project_root / "models" / "rml2018" / "checkpoints"
ckpt_dir.mkdir(exist_ok=True)

# --------------------------------------------------------------
# LOAD BEST CHECKPOINT IF AVAILABLE, OTHERWISE LOAD DEFAULT
# --------------------------------------------------------------

best_ckpt_txt = ckpt_dir / "best_checkpoint.txt"

if best_ckpt_txt.exists():
    with open(best_ckpt_txt, "r") as f:
        best_path = Path(f.read().strip())
    print("Loading best checkpoint:", best_path)
    model = load_model(best_path)
else:
    print("Best checkpoint not found. Loading default model:", default_model_path)
    model = load_model(default_model_path)

# ---------------------------------------------------------
# Dataset loader
# ---------------------------------------------------------
def load_all_rml2018(h5_path, classes_txt, snr_range=(-20,30), max_per_class=3000):
    with open(classes_txt, "r") as f:
        class_list = ast.literal_eval(f.read().split("=")[-1].strip())
        class_list = [c.strip(" '") for c in class_list]

    with h5py.File(h5_path, "r") as f:
        X, Y, Z = f["X"][:], f["Y"][:], f["Z"][:]

    per_class = {cls: [] for cls in class_list}

    for i in range(len(X)):
        snr = int(Z[i][0])
        if snr_range[0] <= snr <= snr_range[1]:
            cls = class_list[int(Y[i].argmax())]
            sig = np.hstack([X[i], np.full((1024,1), snr, dtype=np.float32)])
            per_class[cls].append(sig)

    if max_per_class:
        for k in per_class:
            random.shuffle(per_class[k])
            per_class[k] = per_class[k][:max_per_class]

    X_out, y_out = [], []
    for cls, arr in per_class.items():
        X_out.extend(arr)
        y_out.extend([cls] * len(arr))

    return np.array(X_out, dtype=np.float32), np.array(y_out), class_list

# ---------------------------------------------------------
# Load dataset fresh
# ---------------------------------------------------------
MAX_SAMPLES_PER_CLASS = 3000
SNR_RANGE = (-20, 30)
TEST_SPLIT = 0.20

X_all, y_all, classes = load_all_rml2018(
    H5_PATH, CLASSES_PATH,
    snr_range=SNR_RANGE,
    max_per_class=MAX_SAMPLES_PER_CLASS
)

le = LabelEncoder()
y_all_enc = le.fit_transform(y_all)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all_enc,
    test_size=TEST_SPLIT,
    stratify=y_all_enc,
    random_state=42
)

print("Train:", X_tr.shape, "Test:", X_te.shape)

# ---------------------------------------------------------
# Fine-tune model
# ---------------------------------------------------------
LR = 1e-5
EPOCHS = 5
BATCH = 64

model.compile(
    optimizer=Adam(LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_te, y_te),
    epochs=EPOCHS,
    batch_size=BATCH,
    verbose=2
)

# ---------------------------------------------------------
# Save new checkpoint + metadata
# ---------------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ckpt_name = f"rml2018_lstm_finetuned_{timestamp}.keras"
ckpt_path = ckpt_dir / ckpt_name

model.save(ckpt_path)
print("Saved checkpoint:", ckpt_path)

meta = {
    "timestamp": timestamp,
    "checkpoint": ckpt_name,
    "val_accuracy": float(history.history["val_accuracy"][-1]),
    "train_accuracy": float(history.history["accuracy"][-1]),
    "val_loss": float(history.history["val_loss"][-1]),
    "train_loss": float(history.history["loss"][-1])
}
meta_path = ckpt_dir / f"{ckpt_name}.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=4)
print("Saved metadata:", meta_path)

# ---------------------------------------------------------
# Evaluate ALL checkpoints and select best
# ---------------------------------------------------------
ckpts = sorted(ckpt_dir.glob("rml2018_lstm_finetuned_*.keras"))
results = []

print("\nEvaluating checkpoints:", len(ckpts))

for ckpt in ckpts:
    meta_file = ckpt_dir / f"{ckpt.name}.json"
    if not meta_file.exists():
        continue

    with open(meta_file, "r") as f:
        meta = json.load(f)

    m = load_model(ckpt)
    loss, acc = m.evaluate(X_te, y_te, verbose=0)

    results.append({
        "name": ckpt.name,
        "path": str(ckpt),
        "meta_val_acc": meta["val_accuracy"],
        "eval_acc": float(acc),
        "timestamp": meta["timestamp"]
    })

results_sorted = sorted(results, key=lambda x: x["eval_acc"], reverse=True)
best = results_sorted[0]

print("\nBest checkpoint:")
print(best)

best_file = ckpt_dir / "best_checkpoint.txt"
with open(best_file, "w") as f:
    f.write(best["path"])

print("Best checkpoint saved to:", best_file)

# ---------------------------------------------------------
# Plot improvement over time
# ---------------------------------------------------------
timestamps = [r["timestamp"] for r in results_sorted[::-1]]
val_accs = [r["meta_val_acc"] for r in results_sorted[::-1]]
eval_accs = [r["eval_acc"] for r in results_sorted[::-1]]

plt.figure(figsize=(12,5))
plt.plot(val_accs, marker="o", label="Validation Accuracy (metadata)")
plt.plot(eval_accs, marker="x", label="Eval Accuracy (current test)")
plt.xticks(range(len(timestamps)), timestamps, rotation=45, ha="right")
plt.xlabel("Checkpoint Timeline")
plt.ylabel("Accuracy")
plt.title("Fine-Tuning Improvement Over Time")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()



Loading best checkpoint: /home/rameyjm7/workspace/ML-wireless-signal-classification/models/rml2018_checkpoints/rml2018_lstm_finetuned_20251119_224735.keras
Train: (57600, 1024, 3) Test: (14400, 1024, 3)
Epoch 1/5
900/900 - 72s - 79ms/step - accuracy: 0.5218 - loss: 1.4559 - val_accuracy: 0.5365 - val_loss: 1.4164
Epoch 2/5


KeyboardInterrupt: 